In [ ]:
import jax
import jax.numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P

from sfp.utils import benchmark, numerics
from sfp.kernels.sharded.distributed_gemm import jax_pallas_gemm, ag_gemm_ar_serial, fused_ag_gemm_ar

In [2]:
m, k, n = 16384, 16384, 8192

k1, k2 = jax.random.split(jax.random.key(0), 2)
lhs = jax.random.normal(k1, (m, k), dtype=jnp.bfloat16)
rhs = jax.random.normal(k2, (k, n), dtype=jnp.bfloat16)

In [ ]:
num_devices = jax.device_count()
mesh = jax.make_mesh((2, 2), ("x", "y"))
lhs_sharding = NamedSharding(mesh, P('x', 'y'))
rhs_sharding = NamedSharding(mesh, P('x', None))

lhs = jax.device_put(lhs, lhs_sharding)
rhs = jax.device_put(rhs, rhs_sharding)

In [4]:
def jax_matmul(x: jax.Array, y: jax.Array) -> jax.Array:
    return jnp.matmul(x, y)

ref = jax_matmul(lhs, rhs)

In [ ]:
ref

In [6]:
jax_matmul_compiled = jax.jit(jax_matmul)
jmc = jax_matmul_compiled(lhs, rhs)
jmc.block_until_ready()

with jax.profiler.trace('./traces/naive_matmul'):
    result = jax_matmul_compiled(lhs, rhs)
    result.block_until_ready()

In [ ]:
benchmark(jax_matmul_compiled, lhs, rhs)

In [9]:
jpg = jax.jit(
    jax.shard_map(
        jax_pallas_gemm,
        mesh=mesh,
        in_specs=(P('x', 'y'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False
    )
)

jpg_compiled = jpg.lower(lhs, rhs).compile({'xla_enable_transpose_trace': True})
result = jpg_compiled(lhs, rhs)
result.block_until_ready()

with jax.profiler.trace('./traces/jpg'):
    fc1 = jpg_compiled(lhs, rhs)
    fc1.block_until_ready()

In [ ]:
benchmark(jpg_compiled, lhs, rhs)

In [ ]:
numerics.compare(ref, jpg(lhs, rhs))

In [13]:
agas = jax.jit(
    jax.shard_map(
        ag_gemm_ar_serial,
        mesh=mesh,
        in_specs=(P('x', 'y'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False
    )
)

agas_compiled = agas.lower(lhs, rhs).compile({'xla_enable_transpose_trace': True})
result = agas_compiled(lhs, rhs)
result.block_until_ready()

with jax.profiler.trace('./traces/agas'):
    fc2 = jpg_compiled(lhs, rhs)
    fc2.block_until_ready()

In [ ]:
benchmark(agas_compiled, lhs, rhs)

In [ ]:
numerics.compare(ref, agas(lhs, rhs))

In [16]:
fga = jax.jit(
    jax.shard_map(
        fused_ag_gemm_ar,
        mesh=mesh,
        in_specs=(P('x', 'y'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False
    )
)

fga_compiled = fga.lower(lhs, rhs).compile({'xla_enable_transpose_trace': True})
result = fga_compiled(lhs, rhs)
result.block_until_ready()

with jax.profiler.trace('./traces/fga'):
    fc3 = fga_compiled(lhs, rhs)
    fc3.block_until_ready()

In [ ]:
benchmark(fga_compiled, lhs, rhs)

In [ ]:
numerics.compare(ref, fga(lhs, rhs))